### Filter non-edible products

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
import math
import torch

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA L4


In [4]:
device = 0 if torch.cuda.is_available() else -1

In [5]:
path = "/content/drive/MyDrive/Projects/Haltgut/data/german_foods.csv"
df = pd.read_csv(path)

In [6]:
df["edible"] = None

In [7]:
df = df[df["german_name"] == True]
df.shape

(137969, 6)

In [33]:
batch_size = 8

In [34]:
classifier = pipeline(
    "zero-shot-classification", 
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    batch_size=2*batch_size   
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [42]:
labels = [
    "Lebensmittel für Menschen",
    "kein Lebensmittel"
]

In [13]:
n, _ = df.shape

In [45]:
for i in tqdm(range(0, n, batch_size), total=math.ceil(n/batch_size)):

    ### determine last index of the batch
    end = min(i + batch_size, n)

    ### get the batch from the df
    batch_series = df.iloc[i:end]["product_name"]
    batch = batch_series.to_list()
    ### use the classifier to determine if the product is edible
    result = classifier(
        batch,
        candidate_labels=labels,
        hypothesis_template="Dieser Artikel ist {}.",
        batch_size=64
    )

    ### process the output into labels
    edible = [r['labels'][0] == 'Lebensmittel für Menschen' for r in result]
    ### update the dataframe
    df.loc[batch_series.index, "edible"] = edible

100%|██████████| 17247/17247 [09:30<00:00, 30.25it/s]


In [47]:
### drop unnamed column
df = df.drop(columns=['Unnamed: 0'])

In [48]:
### save the csv
df.to_csv(path, sep=',')

In [49]:
df[df["edible"]==True].shape[0], df[df["edible"]==False].shape[0] 

(117671, 20298)